# 05 Analysis master

**Owner:** Sahil

1. Join cohort + real outcomes + club strength + nation rank into `data/clean/analysis.parquet` (one row per cohort entrant).
2. **Calibration (the story's ending):** what share of each potential band reached its potential in the game, and
   what share "made it" in real life.
3. Compute every headline number the chart titles need and write it to `outputs/findings.json`.
4. Propose candidates for the **featured five** (FIFA 15 entrants where possible). Nothing is finalised here.

Conventions (CLAUDE.md §6): gap = potential at entry − peak later rating (positive = overrated); vanished players have no
gap and stay their own category; already-peaked players are excluded from accuracy metrics. Real-life "made it" =
≥ 3,000 Big-5 league minutes after entry.

**Provisional until the manual reviews are done:** Transfermarkt matches still pending in `reports/match_review.csv`,
and the seeded club-name overrides. Re-run this notebook after the reviews.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config, dq, findings as fd, outcomes

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

CHECKS, NOTES = [], []
def check(name, passed, detail=""):
    CHECKS.append({"Check": name, "Result": "PASS" if passed else "FAIL", "Detail": detail})
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + detail if detail else ""))
    return passed

NB = "05_analysis_master"

def spearman(x, y):
    # Spearman rank correlation on pairwise-complete rows (Pearson on ranks; no scipy needed)
    m = x.notna() & y.notna()
    return x[m].rank().corr(y[m].rank())
R, FS, V = config.STATUS_REACHED, config.STATUS_FELL_SHORT, config.STATUS_VANISHED

## 1. Join into `analysis.parquet`

In [2]:
coh = pd.read_parquet(config.CLEAN / "cohort.parquet")
xw = pd.read_parquet(config.CLEAN / "tm_crosswalk.parquet")
real = pd.read_parquet(config.CLEAN / "real_outcomes.parquet").drop(columns=["entry_date"])
cs = pd.read_parquet(config.CLEAN / "club_strength.parquet")
nr = pd.read_parquet(config.CLEAN / "nation_rank.parquet")
ni = pd.read_parquet(config.CLEAN / "nation_iso.parquet")

a = coh.merge(xw[["fifa_player_id", "tm_player_id", "match_method", "needs_review"]].rename(columns={"fifa_player_id": "player_id"}),
              on="player_id", how="left", validate="one_to_one")
a = a.merge(real.drop(columns=["tm_player_id"]).rename(columns={"fifa_player_id": "player_id"}), on="player_id", how="left",
            validate="one_to_one")
a = a.merge(cs[["player_id", "clubelo_club", "club_elo_at_entry", "club_strength_status", "non_european_club"]],
            on="player_id", how="left", validate="one_to_one")
a = a.merge(nr[["fifa_nation", "edition", "nation_rank"]].rename(columns={"fifa_nation": "nationality_name", "edition": "entry_edition",
                                                                           "nation_rank": "nation_rank_at_entry"}),
            on=["nationality_name", "entry_edition"], how="left", validate="many_to_one")
a = a.merge(ni[["fifa_nation", "iso3", "confederation"]].rename(columns={"fifa_nation": "nationality_name"}),
            on="nationality_name", how="left", validate="many_to_one")

a["is_matched"] = a["tm_player_id"].notna()
# Real-life "made it": unmatched entrants count as not made it. Every player with a Big-5 appearance is in
# Transfermarkt's players table (notebook 02), so an unmatched entrant has no Big-5 minutes unless his match was missed.
for t in config.MADE_IT_MINUTES_SENSITIVITY:
    a[f"made_it_{t}"] = a[f"made_it_{t}"].fillna(False).astype(bool)
a["made_it"] = a[f"made_it_{config.MADE_IT_MINUTES_DEFAULT}"]
a["big5_minutes"] = a["big5_minutes"].fillna(0).where(a["is_matched"])
a["in_accuracy_sample"] = ~a["already_peaked"]

comps = pd.read_csv(config.RAW_TM / "competitions.csv")
current_season = int(pd.read_csv(config.RAW_TM / "players.csv", usecols=["last_season"])["last_season"].max())
a["latest_club_tier"] = outcomes.latest_club_tier(a, comps, current_season)

check("analysis: one row per cohort entrant", len(a) == len(coh) and a["player_id"].is_unique, f"{len(a):,}")
check("Status counts sum to the cohort size", int(a["status"].value_counts().sum()) == len(a))
a.to_parquet(config.CLEAN / "analysis.parquet", index=False)
print("analysis.parquet:", a.shape)

n_peaked = int(a["already_peaked"].sum())
print(f"Already peaked at entry (excluded from accuracy metrics): {n_peaked}")
acc = a[a["in_accuracy_sample"]]

PASS - analysis: one row per cohort entrant | 11,519
PASS - Status counts sum to the cohort size 
analysis.parquet: (11519, 140)
Already peaked at entry (excluded from accuracy metrics): 0


## 2. Calibration: the game vs real life
Shares are of all entrants in the band (vanished players are in the denominator as their own category). Mean gap is over
players with a later edition.

In [3]:
def band_table(df):
    g = df.groupby("potential_band", observed=True)
    t = pd.DataFrame({
        "n": g.size(),
        "reached_%": g["status"].apply(lambda s: (s == R).mean() * 100),
        "fell_short_%": g["status"].apply(lambda s: (s == FS).mean() * 100),
        "vanished_%": g["status"].apply(lambda s: (s == V).mean() * 100),
        "mean_gap": g["gap"].apply(lambda s: s.astype(float).mean()),
        "matched_to_TM_%": g["is_matched"].mean() * 100,
        "made_it_real_%": g["made_it"].mean() * 100,
        "made_it_real_matched_only_%": g.apply(lambda d: d.loc[d["is_matched"], "made_it"].mean() * 100, include_groups=False),
        "made_it_1500_%": g["made_it_1500"].mean() * 100,
        "made_it_5000_%": g["made_it_5000"].mean() * 100,
    })
    t.index = t.index.astype(str)
    return t

cal = band_table(acc)
tot = band_table(acc.assign(potential_band="All"))
cal = pd.concat([cal, tot])
print("=" * 100)
print("CALIBRATION: did the game's potential predict careers?  (cohort entrants FIFA 15–18, aged 21 or under)")
print("=" * 100)
display(cal.round(1))

CALIBRATION: did the game's potential predict careers?  (cohort entrants FIFA 15–18, aged 21 or under)


,n,reached_%,fell_short_%,vanished_%,mean_gap,matched_to_TM_%,made_it_real_%,made_it_real_matched_only_%,made_it_1500_%,made_it_5000_%
potential_band,,,,,,,,,,
<70,5840,29.1,37.3,33.6,1.2,40.7,3.9,9.6,5.1,3.0
70–74,3191,19.7,60.6,19.7,4.5,65.8,9.7,14.7,12.2,7.6
75–79,1784,16.3,75.3,8.5,6.1,81.8,22.4,27.4,26.8,18.0
80–84,600,16.8,81.2,2.0,6.4,90.3,45.8,50.7,52.8,40.0
85+,104,10.6,85.6,3.8,6.8,98.1,68.3,69.6,73.1,61.5
All,11519,23.7,52.3,23.9,3.5,57.1,11.1,19.5,13.5,9.1


In [4]:
# The same question within each band: did 'reached in the game' line up with 'made it' in real life?
xt = (acc[acc["potential"] >= 75].groupby(["potential_band", "status"], observed=True)["made_it"]
         .agg(n="size", made_it_pct=lambda s: round(s.mean() * 100, 1)).unstack("status"))
display(xt)
by_ed = (acc.groupby("entry_edition_label")
            .agg(n=("player_id", "size"), wonderkids=("is_wonderkid", "sum"),
                 reached_pct=("status", lambda s: round((s == R).mean() * 100, 1)),
                 vanished_pct=("status", lambda s: round((s == V).mean() * 100, 1)),
                 made_it_pct=("made_it", lambda s: round(s.mean() * 100, 1))))
display(by_ed)

n                  made_it_pct                 
status         Fell short Reached Vanished  Fell short Reached Vanished
potential_band                                                         
75–79                1343     290      151        13.1    76.9      0.7
80–84                 487     101       12        36.6    96.0      0.0
85+                    89      11        4        67.4   100.0      0.0

,n,wonderkids,reached_pct,vanished_pct,made_it_pct
entry_edition_label,,,,,
FIFA 15,4594,48,34.0,20.2,14.1
FIFA 16,2521,13,22.0,26.8,9.0
FIFA 17,2250,21,15.3,26.4,9.6
FIFA 18,2154,22,12.4,26.0,8.9


## 3. Headline numbers by chart

In [5]:
F = {}
f15 = acc[acc["entry_edition"] == 15]
wk15 = f15[f15["is_wonderkid"]]
F["cohort"] = {
    "entrants": len(a), "entrants_by_edition": a["entry_edition_label"].value_counts().sort_index().to_dict(),
    "already_peaked_excluded": n_peaked, "wonderkids": int(a["is_wonderkid"].sum()),
    "status_counts": a["status"].value_counts().to_dict(), "vanished": int((a["status"] == V).sum()),
    "vanished_pct": fd.pct((a["status"] == V).mean()),
    "reached_pct": fd.pct((acc["status"] == R).mean()), "mean_gap": fd.gap1(acc["gap"].astype(float).mean()),
    "made_it_pct": fd.pct(acc["made_it"].mean()),
}
F["match"] = {
    "matched": int(a["is_matched"].sum()), "match_rate_pct": fd.pct(a["is_matched"].mean()),
    "match_rate_pct_by_band": {str(k): fd.pct(v) for k, v in a.groupby("potential_band", observed=True)["is_matched"].mean().items()},
    "pending_review_players": int(a["needs_review"].fillna(False).sum()),
}
F["calibration_by_band"] = {b: {"n": int(r["n"]), "reached_pct": fd.pct(r["reached_%"] / 100),
                                 "fell_short_pct": fd.pct(r["fell_short_%"] / 100), "vanished_pct": fd.pct(r["vanished_%"] / 100),
                                 "mean_gap": fd.gap1(r["mean_gap"]), "made_it_pct": fd.pct(r["made_it_real_%"] / 100),
                                 "made_it_matched_only_pct": fd.pct(r["made_it_real_matched_only_%"] / 100),
                                 "made_it_1500_pct": fd.pct(r["made_it_1500_%"] / 100), "made_it_5000_pct": fd.pct(r["made_it_5000_%"] / 100)}
                             for b, r in cal.iterrows()}

# Chart 1: FIFA 15 wonderkids
F["chart_01"] = {"fifa15_wonderkids": len(wk15), "reached": int((wk15["status"] == R).sum()),
                 "fell_short": int((wk15["status"] == FS).sum()), "vanished": int((wk15["status"] == V).sum()),
                 "mean_gap": fd.gap1(wk15["gap"].astype(float).mean()),
                 "made_it": int(wk15["made_it"].sum()), "made_it_pct": fd.pct(wk15["made_it"].mean())}

# Chart 2: under-21s with potential >= 85 in each edition (every edition, not first entry)
panel = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["edition", "edition_label", "age", "potential"])
wk_ed = (panel[(panel["age"] <= config.MAX_ENTRY_AGE) & (panel["potential"] >= config.WONDERKID_POTENTIAL)]
         .groupby(["edition", "edition_label"]).size().reset_index(name="wonderkids"))
check("Chart 2 counts match CLAUDE.md §7", wk_ed.set_index("edition")["wonderkids"].to_dict() == config.EXPECTED_U21_POT85_PER_EDITION)
F["chart_02"] = {"by_edition": dict(zip(wk_ed["edition_label"], wk_ed["wonderkids"])),
                 "first": int(wk_ed["wonderkids"].iat[0]), "last": int(wk_ed["wonderkids"].iat[-1]),
                 "max": int(wk_ed["wonderkids"].max()), "max_edition": wk_ed.loc[wk_ed["wonderkids"].idxmax(), "edition_label"],
                 "ratio_max_to_first": round(wk_ed["wonderkids"].max() / wk_ed["wonderkids"].iat[0], 1)}

# Chart 4: potential >= 75 scatter
p75 = acc[(acc["potential"] >= 75) & acc["peak_later"].notna()]
F["chart_04"] = {"n_points": int((acc["potential"] >= 75).sum()), "n_with_later_edition": len(p75),
                 "below_diagonal_pct": fd.pct((p75["peak_later"] < p75["potential"]).mean())}

# Chart 5: calibration ladder (same as calibration_by_band)
F["chart_05"] = {b: F["calibration_by_band"][b] for b in config.BAND_ORDER}
display(pd.DataFrame(F["chart_05"]).T)

PASS - Chart 2 counts match CLAUDE.md §7 


,n,reached_pct,fell_short_pct,vanished_pct,mean_gap,made_it_pct,made_it_matched_only_pct,made_it_1500_pct,made_it_5000_pct
<70,5840.0,29.0,37.0,34.0,1.2,4.0,10.0,5.0,3.0
70–74,3191.0,20.0,61.0,20.0,4.5,10.0,15.0,12.0,8.0
75–79,1784.0,16.0,75.0,8.0,6.1,22.0,27.0,27.0,18.0
80–84,600.0,17.0,81.0,2.0,6.4,46.0,51.0,53.0,40.0
85+,104.0,11.0,86.0,4.0,6.8,68.0,70.0,73.0,62.0


### Chart 6: busts and bloomers
**Busts:** largest gap among **wonderkids** (potential ≥ 85), where the promise was biggest. Ranked by raw gap, the top busts
are obscure lower-league players. **Bloomers:** most negative gap among entrants whose peak later rating reached 80+.
Ties: larger potential first (busts), higher peak first (bloomers), then name.

In [6]:
cols = ["player_id", "short_name", "long_name", "entry_edition_label", "club_name", "nationality_name", "position_group",
        "potential", "peak_later", "gap", "big5_minutes", "made_it"]
busts = (acc[acc["is_wonderkid"] & acc["peak_later"].notna()]
           .sort_values(["gap", "potential", "short_name"], ascending=[False, False, True]).head(10)[cols])
bloomers = (acc[acc["peak_later"] >= 80]
              .sort_values(["gap", "peak_later", "short_name"], ascending=[True, False, True]).head(10)[cols])
print("Top 10 busts"); display(busts)
print("Top 10 bloomers"); display(bloomers)
def plist(df):
    return [{"player_id": int(r.player_id), "player": r.short_name, "entry_edition": r.entry_edition_label,
             "potential": int(r.potential), "peak_later": int(r.peak_later), "gap": int(r.gap)} for r in df.itertuples()]
F["chart_06"] = {"busts": plist(busts), "bloomers": plist(bloomers),
                 "biggest_bust": plist(busts.head(1))[0], "biggest_bloomer": plist(bloomers.head(1))[0]}

Top 10 busts


,player_id,short_name,long_name,entry_edition_label,club_name,nationality_name,position_group,potential,peak_later,gap,big5_minutes,made_it
10968,240000,E. Abouchabaka,Elias Abouchabaka,FIFA 18,RB Leipzig,Germany,MID,85,63,22,0.0,False
1988,212455,H. Mastour,Hachim Mastour,FIFA 16,Málaga,Morocco,MID,87,68,19,5.0,False
7942,231395,José Gomes,José Gomes,FIFA 17,Benfica,Portugal,FWD,85,66,19,NaN,False
8901,234317,V. Thill,Vincent Thill,FIFA 18,Metz,Luxembourg,MID,85,66,19,26.0,False
9270,235266,C. Früchtl,Christian Früchtl,FIFA 18,FC Bayern München,Germany,GK,86,68,18,9.0,False
10646,239249,Oriol Busquets,Oriol Busquets Mas,FIFA 18,FC Barcelona B,Spain,MID,85,67,18,196.0,False
9699,236248,B. Woodburn,Benjamin Luke Woodburn,FIFA 18,Liverpool,Wales,FWD,85,68,17,6.0,False
10361,238232,Han Kwang Song,Kwang Song Han,FIFA 18,Perugia,Korea DPR,FWD,86,70,16,276.0,False
1895,212210,R. Živković,Richairo Živković,FIFA 15,Ajax,Netherlands,FWD,86,70,16,106.0,False
10954,239981,J. Arp,Jann-Fiete Arp,FIFA 18,Hamburger SV,Germany,FWD,85,69,16,1544.0,False


Top 10 bloomers


,player_id,short_name,long_name,entry_edition_label,club_name,nationality_name,position_group,potential,peak_later,gap,big5_minutes,made_it
4906,224179,Borja Iglesias,Borja Iglesias Quintás,FIFA 15,Celta de Vigo,Spain,FWD,55,83,-28,14900.0,True
5542,225375,K. Laimer,Konrad Laimer,FIFA 15,Salzburg,Austria,MID,57,83,-26,13592.0,True
4738,223848,S. Milinković-Savić,Sergej Milinković-Savić,FIFA 15,Genk,Serbia,MID,65,86,-21,21564.0,True
5632,225508,E. Bailly,Eric Bertrand Bailly,FIFA 15,Espanyol,Côte d'Ivoire,DEF,63,84,-21,11441.0,True
11014,240091,G. Vicario,Guglielmo Vicario,FIFA 18,Venezia,Italy,GK,61,82,-21,14934.0,True
3845,221479,D. Calvert-Lewin,Dominic Calvert-Lewin,FIFA 15,Sheffield United,England,FWD,60,81,-21,18492.0,True
3931,221697,O. Watkins,Ollie Watkins,FIFA 15,Exeter City,England,FWD,59,80,-21,18110.0,True
3726,221087,Pau López,Pau López Sabata,FIFA 15,Espanyol,Spain,GK,63,83,-20,22896.0,True
4461,223113,K. Piątek,Krzysztof Piątek,FIFA 16,Zagłębie Lubin,Poland,FWD,60,80,-20,9928.0,True
5091,224458,Diogo Jota,Diogo José Teixeira da Silva,FIFA 15,Paços de Ferreira,Portugal,MID,66,85,-19,11632.0,True


### Chart 7: did EA learn? (learning cohorts, peak within 4 editions)

In [7]:
lc = pd.read_parquet(config.CLEAN / "learning_cohorts.parquet")
lca = lc[~lc["already_peaked"]]
learn = (lca.groupby("entry_edition_label")
            .agg(n=("player_id", "size"),
                 reached_within_4=("status_within_4", lambda s: (s == R).mean()),
                 mean_gap_within_4=("gap_within_4", lambda s: s.astype(float).mean())))
display(learn.round(3))
F["chart_07"] = {"by_entry_edition": {k: {"n": int(r.n), "reached_within_4_pct": fd.pct(r.reached_within_4),
                                          "mean_gap_within_4": fd.gap1(r.mean_gap_within_4)} for k, r in learn.iterrows()},
                 "first_reached_pct": fd.pct(learn["reached_within_4"].iat[0]), "last_reached_pct": fd.pct(learn["reached_within_4"].iat[-1]),
                 "first_mean_gap": fd.gap1(learn["mean_gap_within_4"].iat[0]), "last_mean_gap": fd.gap1(learn["mean_gap_within_4"].iat[-1]),
                 "already_peaked_excluded": int(lc["already_peaked"].sum())}

,n,reached_within_4,mean_gap_within_4
entry_edition_label,,,
FIFA 15,4594,0.250,2.762
FIFA 16,2521,0.152,4.972
FIFA 17,2250,0.098,6.792
FIFA 18,2154,0.083,7.559
FIFA 19,2207,0.076,8.054
FIFA 20,2187,0.059,8.763


### Charts 8–11: who the game misjudged (nation, club strength, position)

In [8]:
g = acc[acc["peak_later"].notna()].copy()
g["gap_f"] = g["gap"].astype(float)

nat = (g.groupby("nationality_name").agg(n=("player_id", "size"), mean_gap=("gap_f", "mean"),
                                         mean_rank=("nation_rank_at_entry", "mean"),
                                         reached=("status", lambda s: (s == R).mean())))
nat = nat[nat["n"] >= config.CHART_MIN_NATION_N].sort_values("mean_gap")
rho_nat = spearman(nat["mean_rank"], nat["mean_gap"])
F["chart_08"] = {"nations": len(nat), "min_n": config.CHART_MIN_NATION_N,
                 "least_overrated": {"nation": nat.index[0], "mean_gap": fd.gap1(nat["mean_gap"].iat[0]), "n": int(nat["n"].iat[0])},
                 "nations_with_positive_mean_gap": int((nat["mean_gap"] > 0).sum()),
                 "most_overrated": {"nation": nat.index[-1], "mean_gap": fd.gap1(nat["mean_gap"].iat[-1]), "n": int(nat["n"].iat[-1])}}
F["chart_09"] = {"nations": len(nat), "spearman_rank_vs_gap": round(rho_nat, 2)}
print(f"Nations n >= {config.CHART_MIN_NATION_N}: {len(nat)}; Spearman(mean nation rank, mean gap) = {rho_nat:.2f}")
display(pd.concat([nat.head(5), nat.tail(5)]).round(2))

clb = (g[g["club_elo_at_entry"].notna()].groupby("club_name")
         .agg(n=("player_id", "size"), mean_gap=("gap_f", "mean"), mean_elo=("club_elo_at_entry", "mean")))
clb = clb[clb["n"] >= config.CHART_MIN_CLUB_N]
rho_club = spearman(clb["mean_elo"], clb["mean_gap"])
rho_club_players = spearman(g["club_elo_at_entry"], g["gap_f"])
F["chart_10"] = {"clubs": len(clb), "min_n": config.CHART_MIN_CLUB_N, "spearman_elo_vs_gap_clubs": round(rho_club, 2),
                 "spearman_elo_vs_gap_players": round(rho_club_players, 2)}
print(f"Clubs n >= {config.CHART_MIN_CLUB_N} with Elo: {len(clb)}; Spearman(club Elo, mean gap) = {rho_club:.2f} "
      f"(player level {rho_club_players:.2f})")

pos = (g.groupby("primary_position").agg(n=("player_id", "size"), mean_potential=("potential", "mean"),
                                         mean_peak=("peak_later", lambda s: s.astype(float).mean()), mean_gap=("gap_f", "mean"))
         .sort_values("mean_gap"))
display(pos.round(2))
F["chart_11"] = {"positions": {p: {"n": int(r.n), "mean_potential": int(round(r.mean_potential)),
                                   "mean_peak_later": int(round(r.mean_peak)), "mean_gap": fd.gap1(r.mean_gap)} for p, r in pos.iterrows()},
                 "most_overrated_position": pos.index[-1], "least_overrated_position": pos.index[0]}

Nations n >= 15: 56; Spearman(mean nation rank, mean gap) = 0.04


,n,mean_gap,mean_rank,reached
nationality_name,,,,
Mali,21,0.67,60.19,0.52
Bosnia and Herzegovina,23,0.83,23.87,0.52
Denmark,196,0.98,33.4,0.48
Albania,17,1.12,53.41,0.47
Côte d'Ivoire,50,1.14,33.32,0.42
Russia,118,5.79,30.44,0.19
Georgia,17,5.88,115.18,0.12
South Africa,20,5.90,66.1,0.15
Turkey,143,6.31,32.47,0.17


Clubs n >= 5 with Elo: 397; Spearman(club Elo, mean gap) = 0.04 (player level 0.02)


,n,mean_potential,mean_peak,mean_gap
primary_position,,,,
RB,586,69.46,66.93,2.53
CDM,497,70.81,68.23,2.58
LB,622,69.81,67.17,2.64
CB,1279,70.29,67.34,2.95
CM,1229,69.72,66.43,3.29
LM,452,71.00,67.67,3.33
RM,437,71.23,67.72,3.51
GK,917,68.28,64.59,3.69
LW,237,74.32,70.22,4.10


### Chart 12: which teenage attributes mattered (outfield players)

In [9]:
skip = {"goalkeeping_diving", "goalkeeping_handling", "goalkeeping_kicking", "goalkeeping_positioning",
        "goalkeeping_reflexes", "goalkeeping_speed"}
attr_cols = config.FACE_STATS + [c for c in coh.columns if c.split("_")[0] in
            {"attacking", "skill", "movement", "power", "mentality", "defending"} and c not in skip]
attr_cols = list(dict.fromkeys(attr_cols + ["weak_foot", "skill_moves"]))   # in-game rated attributes only, no duplicates
other_cols = ["age", "height_cm", "weight_kg"]                                 # not FIFA attributes: reported separately
o = g[~g["is_gk"]].copy()
o["beat_potential"] = -o["gap_f"]   # peak later - potential
corr = o[attr_cols].apply(lambda s: spearman(s, o["beat_potential"])).sort_values()
other = o[other_cols].apply(lambda s: spearman(s, o["beat_potential"]))
check("Chart 12: attribute list has no duplicates", len(attr_cols) == len(set(attr_cols)), f"{len(attr_cols)} attributes")
display(corr.round(3).to_frame("spearman"))
print("Not attributes (reported separately):"); display(other.round(3).to_frame("spearman"))
F["chart_12"] = {"n_outfield": len(o), "n_attributes": len(attr_cols),
                 "top_positive": {k: round(v, 2) for k, v in corr.tail(8)[::-1].items()},
                 "top_negative": {k: round(v, 2) for k, v in corr.head(8).items()},
                 "max_abs_spearman": round(float(corr.abs().max()), 2),
                 "non_attribute_spearman": {k: round(v, 2) for k, v in other.items()}}

PASS - Chart 12: attribute list has no duplicates | 37 attributes


,spearman
skill_moves,-0.100
attacking_finishing,-0.096
shooting,-0.091
mentality_penalties,-0.090
skill_dribbling,-0.085
dribbling,-0.082
attacking_volleys,-0.081
mentality_positioning,-0.072
movement_balance,-0.070
skill_curve,-0.069


Not attributes (reported separately):


,spearman
age,0.202
height_cm,0.073
weight_kg,0.072


### Chart 14 (provisional tier rule, notebook 12 owns the chart)

In [10]:
tiers = acc[acc["potential"] >= 75].groupby(["potential_band", "status", "latest_club_tier"], observed=True).size()
tier_share = (acc[acc["potential"] >= 75]["latest_club_tier"].value_counts(normalize=True)
                .reindex(outcomes.TIER_ORDER).fillna(0))
display(tier_share.round(3).to_frame("share of potential-75+"))
wk_tier = acc[acc["is_wonderkid"]]["latest_club_tier"].value_counts().reindex(outcomes.TIER_ORDER).fillna(0).astype(int)
F["chart_14"] = {"potential_75plus_tier_pct": {k: fd.pct(v) for k, v in tier_share.items()},
                 "wonderkids_by_tier": wk_tier.to_dict(), "current_season": current_season, "provisional": True}

,share of potential-75+
latest_club_tier,
Big-5,0.172
Other European top flight,0.146
Other or lower league,0.406
Outside Europe,0.097
No current club,0.025
Not in Transfermarkt,0.154


## 4. Featured five: candidates (for team approval; nothing is finalised)
From FIFA 15 entrants where possible:
1. a star who reached his potential; 2. the biggest bust; 3. a late bloomer (potential < 80, peak later ≥ 85);
4. a player whose latest club is outside Europe; 5. a player whose peak later rating equals his potential exactly.

In [11]:
show = ["player_id", "short_name", "long_name", "club_name", "nationality_name", "age", "overall", "potential", "peak_later",
        "gap", "big5_minutes", "peak_market_value_eur", "tm_current_club_name", "latest_club_tier", "match_method"]
f15a = a[a["entry_edition"] == 15].copy()
f15a["peak_market_value_eur_m"] = (f15a["peak_market_value_eur"] / 1e6).round(1)
show = [c if c != "peak_market_value_eur" else "peak_market_value_eur_m" for c in show]
cands = {
    "1 Star who reached his potential": f15a[f15a["is_wonderkid"] & (f15a["status"] == R)].sort_values(["peak_later", "potential"], ascending=False),
    "2 Biggest bust": f15a[f15a["is_wonderkid"] & f15a["peak_later"].notna()].sort_values(["gap", "potential"], ascending=False),
    "3 Late bloomer": f15a[(f15a["potential"] < 80) & (f15a["peak_later"] >= 85)].sort_values(["peak_later", "gap"], ascending=[False, True]),
    "4 Latest club outside Europe": f15a[(f15a["latest_club_tier"] == outcomes.TIER_OUTSIDE)].sort_values(["potential", "peak_later"], ascending=False),
    "5 Peak later = potential": f15a[(f15a["gap"] == 0)].sort_values(["potential", "peak_later"], ascending=False),
}
for k, df in cands.items():
    print(f"\n{k}: {len(df)} FIFA 15 candidates (top 6)")
    display(df[show].head(6))
F["featured_candidates"] = {k: [{"player_id": int(r.player_id), "player": r.short_name} for r in df.head(6).itertuples()]
                            for k, df in cands.items()}
F["featured_candidates"]["status"] = "proposed, awaiting team approval"


1 Star who reached his potential: 7 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
49,192505,R. Lukaku,Romelu Lukaku Menama,Everton,Belgium,21,80,86,88,-2,28669.0,100.0,SSC Napoli,Big-5,auto
373,202652,R. Sterling,Raheem Sterling,Liverpool,England,19,80,86,88,-2,24283.0,160.0,Feyenoord Rotterdam,Other European top flight,auto
1026,207865,Marquinhos,Marcos Aoás Corrêa,Paris Saint Germain,Brazil,20,78,86,88,-2,24303.0,75.0,Paris Saint-Germain,Big-5,auto
140,199556,M. Verratti,Marco Verratti,Paris Saint Germain,Italy,21,80,86,87,-1,15959.0,75.0,Paris Saint-Germain,Other or lower league,auto
251,201399,M. Icardi,Mauro Emanuel Icardi Rivero,Inter,Argentina,21,76,86,87,-1,16942.0,100.0,Galatasaray,Other European top flight,auto
1298,209658,L. Goretzka,Leon Christoph Goretzka,Schalke 04,Germany,19,73,85,87,-2,20817.0,70.0,Bayern Munich,Big-5,auto



2 Biggest bust: 47 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
1895,212210,R. Živković,Richairo Živković,Ajax,Netherlands,17,69,86,70,16,106.0,4.0,FC Emmen,Other or lower league,auto
3390,220048,S. Scuffet,Simone Scuffet,Udinese,Italy,18,71,87,75,12,6733.0,6.0,Pisa Sporting Club,Big-5,auto
131,199151,J. Iturbe,Juan Manuel Iturbe Arévalo,Roma,Argentina,21,77,88,77,11,1035.0,19.0,Aris Thessalonikis,Other or lower league,auto
419,203038,Lucas Piazón,Gustavo Lucas Domingues Piazón,Eintracht Frankfurt,Brazil,20,76,86,75,11,871.0,5.0,Avs Futebol,Other or lower league,auto
1187,208790,Z. Bakkali,Zakaria Bakkali,PSV,Belgium,18,71,85,74,11,1743.0,4.0,RKC Waalwijk,No current club,auto
1651,211176,R. Kishna,Ricardo Dennie Kishna,Ajax,Netherlands,19,73,85,75,10,1217.0,4.0,ADO Den Haag,Other or lower league,auto



3 Late bloomer: 14 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
320,202126,H. Kane,Harry Kane,Tottenham Hotspur,England,20,71,78,90,-12,33891.0,150.0,Bayern Munich,Big-5,auto
1394,210257,Ederson,Ederson Santana de Moraes,Rio Ave,Brazil,20,69,77,89,-12,24516.0,70.0,Manchester City,Big-5,auto
2025,212622,J. Kimmich,Joshua Walter Kimmich,RB Leipzig,Germany,19,63,78,89,-11,25988.0,90.0,Bayern Munich,Big-5,auto
2722,215698,M. Maignan,Mike Maignan,Paris Saint Germain,France,18,56,74,87,-13,27307.0,45.0,AC Milan,Big-5,auto
4738,223848,S. Milinković-Savić,Sergej Milinković-Savić,Genk,Serbia,19,57,65,86,-21,21564.0,90.0,Al-Hilal Saudi Football Club,Outside Europe,auto
4208,222492,L. Sané,Leroy Aziz Sané,Schalke 04,Germany,18,60,74,86,-12,18577.0,100.0,Galatasaray,Other European top flight,auto



4 Latest club outside Europe: 272 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
368,202556,M. Depay,Memphis Depay,PSV,Netherlands,20,78,88,85,3,15013.0,55.0,Sport Club Corinthians Paulista,Outside Europe,auto
216,200949,Lucas Moura,Lucas Rodrigues Moura da Silva,Paris Saint Germain,Brazil,21,82,87,83,4,14251.0,40.0,São Paulo Futebol Clube,Outside Europe,auto
2557,214997,Á. Correa,Ángel Martín Correa,San Lorenzo,Argentina,19,75,86,83,3,16432.0,45.0,Tigres UANL,Outside Europe,auto
767,205525,Bernard,Bernard Anício Caldeira Duarte,Shakhtar Donetsk,Brazil,21,80,86,80,6,3856.0,20.0,Clube Atlético Mineiro,Outside Europe,auto
1464,210513,J. Quintero,Juan Fernando Quintero Paniagua,Porto,Colombia,21,78,86,80,6,817.0,15.0,Club Atlético River Plate,Outside Europe,auto
957,207441,L. Vietto,Luciano Darío Vietto,Villarreal,Argentina,20,73,85,79,6,4050.0,20.0,Club Atlético San Lorenzo de Almagro,Outside Europe,auto



5 Peak later = potential: 234 FIFA 15 candidates (top 6)


,player_id,short_name,long_name,club_name,nationality_name,age,overall,potential,peak_later,gap,big5_minutes,peak_market_value_eur_m,tm_current_club_name,latest_club_tier,match_method
1885,212188,T. Werner,Timo Werner,VfB Stuttgart,Germany,18,70,85,85,0,21328.0,80.0,RB Leipzig,Big-5,auto
680,204970,F. Thauvin,Florian Tristan Mariano Thauvin,Olympique de Marseille,France,21,77,84,84,0,21551.0,50.0,RC Lens,Big-5,auto
1989,212462,Alex Telles,Alex Nicolao Telles,Galatasaray,Brazil,21,72,84,84,0,5355.0,40.0,S. A. F. Botafogo,Outside Europe,auto
1101,208333,E. Can,Emre Can,Liverpool,Germany,20,74,83,83,0,22418.0,40.0,Borussia Dortmund,Big-5,auto
2066,212814,João Mário,João Mário Naval da Costa Eduardo,Sporting CP,Portugal,21,73,83,83,0,4658.0,35.0,Athlitiki Enosi Konstantinoupoleos,Other European top flight,auto
2162,213331,J. Tah,Jonathan Tah,Fortuna Düsseldorf,Germany,18,68,83,83,0,26865.0,40.0,Bayern Munich,Big-5,auto


## 5. Write `outputs/findings.json` and the data-quality section

In [12]:
provisional = {"match_review_pending_players": F["match"]["pending_review_players"],
               "club_overrides_seeded_unreviewed": True,
               "note": "Re-run after the Transfermarkt match review and the club-override check."}
for k, v in F.items():
    fd.update(k, v, NB)
fd.update("provisional", provisional, NB)
print("findings.json sections:", sorted(fd.load().keys()))

checks = pd.DataFrame(CHECKS)
body = f'''
- analysis.parquet: {len(a):,} rows (one per cohort entrant), {a.shape[1]} columns.
- Already-peaked entrants excluded from accuracy metrics: {n_peaked}.
- Real-life "made it" counts unmatched entrants as not made it ({int((~a["is_matched"]).sum()):,} unmatched).
- Provisional: {F["match"]["pending_review_players"]} players still pending Transfermarkt match review.

### Calibration

{dq.df_to_md(cal.round(1).reset_index().rename(columns={"index": "Band", "potential_band": "Band"}))}

### Checks

{dq.df_to_md(checks)}
'''
dq.write_section("05", "05 Analysis master", body)

findings.json sections: ['_meta', 'calibration_by_band', 'chart_01', 'chart_02', 'chart_04', 'chart_05', 'chart_06', 'chart_07', 'chart_08', 'chart_09', 'chart_10', 'chart_11', 'chart_12', 'chart_14', 'cohort', 'featured_candidates', 'match', 'provisional']


## 6. Summary

In [13]:
display(checks)
assert (checks["Result"] == "PASS").all(), checks[checks["Result"] == "FAIL"]

,Check,Result,Detail
0,analysis: one row per cohort entrant,PASS,"11,519"
1,Status counts sum to the cohort size,PASS,
2,Chart 2 counts match CLAUDE.md §7,PASS,
3,Chart 12: attribute list has no duplicates,PASS,37 attributes
